[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/byu-matrix-lab/torchlingo/blob/main/docs/docs/course/lecture-13-back-translation.ipynb)


# Lecture 13: back-translation, training data from text in one language

A parallel corpus is the scarce thing; text in your language alone is far easier to find.
Back-translation turns the second into the first. Translate a pile of sentences in your language
into English, with a model going the *reverse* way, and add those pairs to your training data:
the English side is machine-made, but the side your model learns to *write* is real. Then
retrain English to your language, and measure whether it helped. That is Assignment 13, run as a
small version of the Final Project's Path A: a hypothesis stated first, one change against a named
baseline (your A9 system), the same test set, and a test of whether the difference is real.

By the end you will be able to:

1. say what makes a language low-resource beyond its count of sentence pairs, from what you met
   finding text for this assignment;
2. use monolingual text to make up for scarce parallel data: back-translate it with a reverse
   model (your own, or NLLB-200 where it covers your language), train on the synthetic pairs, and
   measure whether it helped;
3. tell how much of a change came from the new training data and how much from the decoding
   method (greedy or beam search), and test whether a difference is real.

These serve Lecture 13's outcome, which is these three together.

You write the two steps that are the method, reversing the split and combining the data without
contaminating your test set, a hypothesis and a prediction; checks tell you when each is right,
and the notebook runs the rest.

**What you need.** Your A9 files in `MyDrive/CS479/assignment8`, as Lecture 9's Part B left
them: the split, `spm.model`, `a9-settings.json`, `best-a9/` and `test-a9.hyp`. Sentences in your
language, one per line, that are **not** in your train, validation or test sets, ideally as many
as your training set has pairs; Step 1 says where to find them.

**This is long.** Two trainings: the reverse model (none with NLLB), then English to your
language once, on up to twice A9's data. Plan for about three times your A8 training, two with
NLLB. Use an A100, L4 or G4. Every step saves to Drive and can be picked up in a new session: run
the two setup cells, then the step you were on. **When you finish, disconnect the GPU**: Runtime →
Disconnect and delete runtime. Not while a training or back-translation cell is running; the last
cell says why.

<!-- GENERATED:purpose - change the torchlingo metadata, not this cell -->

**CS 479 course notebook** — homework for Lecture 13 (*Strategies for NMT of Low-Resource Languages*). A head start on **A13**.

In [ ]:
# Install TorchLingo in Colab. Nothing to uncomment. Pip's "you may need to restart the
# kernel" does not apply to a first install; if the install fails, the import below stops here.
import sys
if "google.colab" in sys.modules:
    %pip install --quiet "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
    # pip keeps a TorchLingo already in this session when its version number matches, even if
    # it is older code; this swaps in the course branch's, leaving the dependencies alone.
    %pip install --quiet --force-reinstall --no-deps "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
import torchlingo


In [ ]:
# Setup: the GPU, Drive, your A9 files, the configuration every training uses, and the helpers
# the steps below share. Run it first in every session.
import json
import random

import pandas as pd
import sacrebleu
import torch
import torchlingo
from torchlingo.checkpoint import load_checkpoint
from torchlingo.colab import course_dir, setup
from torchlingo.config import Config
from torchlingo.data_processing import create_dataloaders
from torchlingo.data_processing.vocab import SentencePieceVocab
from torchlingo.inference import translate_lines_to_file, translate_to_file
from torchlingo.models import SimpleTransformer
from torchlingo.training import train_model
from torchlingo.training_checkpoint import TrainingCheckpointer

env = setup(gpu=True, drive=True)       # stops with instructions if there is no GPU
device = env.device
# Stops, naming each missing file, if A9 has not left them in MyDrive/CS479/assignment8.
A8_DIR = course_dir(
    env.drive_dir, "assignment8",
    needs=["train.tsv", "val.tsv", "test.tsv", "spm.model", "a9-settings.json",
           "best-a9/model_best.pt", "test-a9.hyp"],
    hint="Lecture 9's Part B writes them; finish A9 first.",
)

settings = json.loads((A8_DIR / "a9-settings.json").read_text(encoding="utf-8"))
NO_SPACES = settings["no_spaces"]
MAX_DECODE = settings["max_decode_length"]
TOKENIZE = "flores200" if NO_SPACES else "13a"     # the BLEU A9 was scored with
SPM = str(A8_DIR / "spm.model")
vocab = SentencePieceVocab(SPM)         # one vocabulary for both languages, so either direction

# Every training runs as many epochs as A9 did, read from A9's checkpoint on Drive, so that the
# added data is the only difference.
a9_run = TrainingCheckpointer(experiment_name="assignment-9", verbose=False)
if a9_run.has_checkpoint():
    a9_state = torch.load(a9_run.path_for("latest"), map_location="cpu", weights_only=False)["state"]
    EPOCHS = len(a9_state["train_losses"])
else:
    EPOCHS = None   # <- only if no A9 checkpoint is found: the epochs your A9 notebook trained for
assert EPOCHS, (
    f"No A9 checkpoint at {a9_run.path_for('latest')}. Set EPOCHS on the line above to the "
    "number of epochs your A9 notebook trained for, and run this cell again."
)
SEED = 479
config = Config(
    d_model=512,
    n_heads=8,
    num_encoder_layers=6,
    num_decoder_layers=6,
    d_ff=2048,
    batch_size=64,
    num_workers=0,
    max_decode_length=MAX_DECODE,
)
NOTES = A8_DIR / "a13-notes.json"   # what each step saves for the write-up


def load_notes():
    """Everything the steps have saved so far, so a dropped session loses none of it."""
    return json.loads(NOTES.read_text(encoding="utf-8")) if NOTES.exists() else {}


def save_note(**items):
    """Add items to a13-notes.json on Drive."""
    NOTES.write_text(json.dumps({**load_notes(), **items}, indent=2, ensure_ascii=False),
                     encoding="utf-8")


def read_tsv(name):
    return pd.read_csv(A8_DIR / name, sep="\t", dtype=str, keep_default_na=False)


def new_model():
    torch.manual_seed(SEED)
    return SimpleTransformer(src_vocab_size=len(vocab), tgt_vocab_size=len(vocab), config=config).to(device)


def train(train_tsv, val_tsv, run_name, best_dir):
    """A9's training call on new files; resumes from its checkpoint if the session dropped."""
    random.seed(SEED)
    train_loader, val_loader, _, _ = create_dataloaders(
        A8_DIR / train_tsv, A8_DIR / val_tsv, use_bucketing=True, num_workers=0, config=config,
        use_sentencepiece=True, sp_model_path=SPM, sp_tgt_model_path=SPM,
    )
    model = new_model()
    result = train_model(model, train_loader, val_loader=val_loader, num_epochs=EPOCHS,
                         device=device, config=config, gradient_clip=1.0, save_dir=A8_DIR / best_dir,
                         log_every=50, checkpointer=TrainingCheckpointer(experiment_name=run_name))
    print(f"Epochs: {len(result.train_losses)}   final val loss {result.val_losses[-1]:.4f}")


def load_best(best_dir):
    model = new_model()
    model.load_state_dict(load_checkpoint(A8_DIR / best_dir / "model_best.pt", map_location=device)["model_state_dict"])
    return model.eval()


_nllb = {}


def back_translate(sentences, path, method, overwrite=False):
    """Translate sentences in your language into English with Step 2's back-translator.

    Writes one line per sentence to path on Drive as it goes, and resumes there if the session
    drops. method is "greedy" or "sample" (or "beam", "beam+noise"), seeded with SEED.
    """
    notes = load_notes()
    if notes["back_translator"] == "nllb":
        from torchlingo_harness.nllb import load_nllb   # an accessory shipped with TorchLingo

        if "model" not in _nllb:
            _nllb["model"] = load_nllb(notes["nllb_code"], "eng_Latn", device=device)
        nllb = _nllb["model"]
        options = {"decode_strategy": method, "beam_size": 5, "top_k": notes.get("top_k")}
        return translate_lines_to_file(
            lambda chunk, first: nllb(chunk, **options, seed=[SEED + first + i for i in range(len(chunk))]),
            sentences, path, nllb.settings(**options, seed=SEED), overwrite=overwrite)
    return translate_to_file(load_best("best-a13-reverse"), sentences, vocab, vocab, path,
                             decode_strategy=method, beam_size=5, top_k=notes.get("top_k"),
                             seed=SEED, device=device, config=config, overwrite=overwrite)


train_df, val_df, test_df = read_tsv("train.tsv"), read_tsv("val.tsv"), read_tsv("test.tsv")
print(f"Your split: {len(train_df):,} train, {len(val_df):,} validation, {len(test_df):,} test pairs.")
print(f"A9 trained for {EPOCHS} epochs; every A13 training runs the same.")

## Step 1: the monolingual text

The sentences you back-translate must be new to every part of your split. One that is already in
your test set would put its reference into training, and the score would measure memory. The cell
removes any that are, removes repeats, runs the contamination check against your test set, and
keeps up to as many as your training set has pairs, so each new training set is up to twice the
old one.

**Where to find it, in this order.** A low-resource language may have little text left over after
A4 and A5, so look in both places:

1. **The Church data first.** Your translation memory holds more than Assignment 5 kept. Rerun
   your Assignment 4 and 5 extraction on the segments it left out, and keep only your language's
   side, one sentence per line, as `mono-tm.txt`. **Not** the target side of your A8 training
   data: back-translating sentences the model already saw paired adds almost nothing, and the
   cell removes them anyway.
2. **A public monolingual set if that runs short**, as `mono-public.txt`:
   [OPUS](https://opus.nlpl.eu), searched by your language's code (its monolingual downloads),
   Wikipedia in your language, or [MADLAD-400](https://huggingface.co/datasets/allenai/MADLAD-400)
   or CC-100 on the [Hugging Face Hub](https://huggingface.co/datasets).

**Mind the domain.** Your test set is Church text. Web text, news or Wikipedia is a different
domain, with other words and other sentences, and synthetic pairs from it may help less on a
Church test set, or hurt. The write-up asks for a sentence on yours.

Both go in `MyDrive/CS479/assignment8`, in UTF-8. **There is no minimum.** The cell prints how
many it used from each file and saves the counts for the write-up, whose fifth question asks
where your text came from and, if it fell short, why.

In [ ]:
from torchlingo.diagnostics import check_contamination

# Your files, in MyDrive/CS479/assignment8, in the order they are used: the Church TM first, and
# the public text only to top it up. Leave out a file you do not have.
MONO_FILES = {
    "Church TM": A8_DIR / "mono-tm.txt",       # <- your language's side of TM segments A5 left out
    "public text": A8_DIR / "mono-public.txt",  # <- optional: OPUS, Wikipedia, MADLAD-400, CC-100
}

found = {source: path for source, path in MONO_FILES.items() if path.exists()}
if not found:
    raise SystemExit(
        f"Put a file of sentences in your language in {A8_DIR}, one per line: mono-tm.txt, and "
        "mono-public.txt if you need more. The markdown above says where to find them. Then run "
        "this cell again."
    )

# Sentences already in your split are removed, and repeats, within a file and across the two.
in_split = set(train_df.tgt) | set(val_df.tgt) | set(test_df.tgt)
mono, used_by_source, seen = [], {}, set()
for source, path in found.items():
    lines = [s.strip() for s in path.read_text(encoding="utf-8-sig").rstrip("\n").split("\n")]
    fresh = [s for s in dict.fromkeys(lines) if s and s not in in_split and s not in seen]
    seen.update(fresh)
    random.Random(SEED).shuffle(fresh)
    take = fresh[:len(train_df) - len(mono)]   # up to as many as your training pairs, in all
    mono += take
    used_by_source[source] = len(take)
    print(f"{source}: {len(lines):,} lines in {path.name}; {len(fresh):,} new to your split; "
          f"using {len(take):,}.")

# The check that matters: no test reference may become a training target.
result = check_contamination(test_df.tgt, mono, name="test")
print(result)
assert result, "a test reference is in the monolingual text; it would put the answer into training"

print(f"\nBack-translating {len(mono):,} sentences, against {len(train_df):,} original training pairs.")
if len(mono) < len(train_df):
    print("Fewer than your training pairs. There is no minimum: the write-up's fifth question asks "
          "how many you found, where, and why not more.")
(A8_DIR / "mono-used.txt").write_text("\n".join(mono) + "\n", encoding="utf-8")
save_note(mono_sources=used_by_source)   # on Drive for the write-up, so a dropped session keeps it

## Step 2: the back-translator, your language to English

Something must translate your monolingual sentences into English. You have two choices:

- **Your own reverse model** (`"own"`): A9's configuration and vocabulary, trained in the other
  direction on your split. It is the first long training. Keep it: `best-a13-reverse/` stays on
  Drive, and A14 compares a model that translates both ways with it.
- **NLLB-200** (`"nllb"`), Meta's model for 200 languages, **if it covers yours**: one training
  fewer. Find your language's code in the
  [FLORES-200 list](https://github.com/facebookresearch/flores/blob/main/flores200/README.md),
  such as `tel_Telu` for Telugu. If it is not there, NLLB does not cover it, and the first cell
  that loads NLLB stops and says so. NLLB is a much larger model trained on far more data than
  yours, so its English is likely better. Whether that makes better training data is part of what
  you find out.

**Yours to write: `reverse`**, whichever you choose. It takes a table of pairs and returns the
same pairs in the other direction, your language as the source and English as the target. The
SentencePiece model you trained in Lecture 9 covers both languages, so that swap is the whole
change; the configuration and vocabulary stay A9's. The reversed test set is how the
back-translator is scored, whichever it is. The cell checks your function on a case whose answer
is known, then on all three files of your split, before it writes them.

With `"own"`, the cell after it is the first long training. If the session drops, run it again
and it resumes.

In [ ]:
BACK_TRANSLATOR = "own"   # <- "own" trains your reverse model; "nllb" uses NLLB-200, if it covers your language
NLLB_CODE = None          # <- with "nllb": your language's FLORES-200 code, such as "tel_Telu"

assert BACK_TRANSLATOR in ("own", "nllb"), 'BACK_TRANSLATOR is "own" or "nllb"'
if BACK_TRANSLATOR == "nllb":
    assert NLLB_CODE, ('Set NLLB_CODE to your language\'s FLORES-200 code, such as "tel_Telu", from '
                       "the list linked above. If your language is not in it, NLLB does not cover it.")
save_note(back_translator=BACK_TRANSLATOR, nllb_code=NLLB_CODE)
print(f"Back-translator: {'NLLB-200, from ' + NLLB_CODE if BACK_TRANSLATOR == 'nllb' else 'your own reverse model'}.")

In [ ]:
def reverse(frame):
    """Return the same pairs in the other direction: your language as src, English as tgt.

    TODO: write it, in a line or two. Keep the two columns named src and tgt, in that order.
    """
    raise NotImplementedError("write reverse, then run this cell again")


# A case whose answer you know, before trusting it on your split.
toy = reverse(pd.DataFrame({"src": ["good morning"], "tgt": ["buenos días"]}))
assert list(toy.columns) == ["src", "tgt"], "keep the columns named src and tgt, in that order"
assert toy.values.tolist() == [["buenos días", "good morning"]], "reverse is not swapping yet"

for name, frame in (("train", train_df), ("val", val_df), ("test", test_df)):
    flipped = reverse(frame)
    assert list(flipped.src) == list(frame.tgt) and list(flipped.tgt) == list(frame.src), (
        f"reversing changed the {name} pairs, not just their direction")
    flipped.to_csv(A8_DIR / f"{name}-reversed.tsv", sep="\t", index=False)
print("Checked and written: train-reversed.tsv, val-reversed.tsv, test-reversed.tsv")

In [ ]:
# The first long training, your language to English, unless NLLB back-translates for you. If
# the session drops, run this again and it resumes.
if load_notes()["back_translator"] == "own":
    train("train-reversed.tsv", "val-reversed.tsv", "assignment-13-reverse", "best-a13-reverse")
else:
    print("NLLB back-translates, so there is no reverse model to train.")

How good is the back-translator? Its English becomes the source side of half your new training
data, so its quality bounds what back-translation can do. The cell scores it on your test set,
reversed, decoding greedily. Expect it to be below your English-to-X score, or above it; either
is worth a sentence in the write-up.

In [ ]:
reverse_test = read_tsv("test-reversed.tsv")
hyp_reverse = back_translate(list(reverse_test.src), A8_DIR / "test-reversed-greedy.hyp", "greedy")
bleu_reverse = sacrebleu.corpus_bleu(hyp_reverse, [list(reverse_test.tgt)])   # English: default BLEU
print(f"Your language to English, on your test set: {bleu_reverse}")
save_note(bleu_reverse=round(bleu_reverse.score, 2))

## Step 3: back-translate, by sampling

How the synthetic English is made matters. **Greedy** decoding would give the back-translator's
most likely English: clean, but regular, so it teaches the forward model little it does not know.
**Sampling** draws each word at random in proportion to its probability, so the English varies
the way real sources do, mistakes included. Edunov et al. (2018) found that sampled
back-translations train a better model than greedy or beam ones, *in all but resource-poor
settings*, where the back-translator is too weak for its samples to be worth much. Many of our
languages are resource-poor, so for yours it is an open question; the write-up asks what your
result suggests.

So this step samples. It runs batched, about 12 minutes for 100,000 sentences at A9's size on a
laptop GPU, less on Colab's. The cell then makes five of the same sentences greedily as well and
prints them side by side, so you can see what sampling changes.

**If NLLB is your back-translator, set `TOP_K = 10`.** NLLB's vocabulary covers 200 languages,
and a draw from all of it often wanders into another script mid-sentence; drawing from the 10
likeliest words, which Edunov et al. also report, keeps the variety and drops the strays. With
your own model, whose vocabulary is only your two languages, leave it at `None`.

It writes `mono-used.bt-sample.en` on Drive as it goes; if the session drops, run the cell again
and it carries on from the last finished chunk.

In [ ]:
TOP_K = None   # <- None draws from every token; with NLLB, 10 (see above)

save_note(bt_method="sample", top_k=TOP_K)
mono = (A8_DIR / "mono-used.txt").read_text(encoding="utf-8").splitlines()
print(f"=== {len(mono):,} sentences into mono-used.bt-sample.en ===")
back = back_translate(mono, A8_DIR / "mono-used.bt-sample.en", "sample")

# Five of them made greedily too, for comparison: the sampled English is what you train on.
shown = mono[:5]
greedy = back_translate(shown, A8_DIR / "mono-shown.bt-greedy.en", "greedy", overwrite=True)
for real, sampled, likeliest in zip(shown, back, greedy):
    print(f"\nreal:     {real}\nsampled:  {sampled}\ngreedy:   {likeliest}")

## Step 4: combine, state a hypothesis, then retrain English to your language

**Yours to write: `combine`.** A new training set is your A9 training pairs, unchanged and first,
then one pair per back-translated sentence: the back-translator's English as the source, your real
sentence as the target. Two kinds of new pair must be left out: one whose English came back
empty, and one whose English is exactly a source sentence of your validation or test set. The
second is easy to miss and is the one that matters: training on a test sentence's source makes
the test score measure memory. The cell checks your function on a known case, then builds the
training set, `train-a13.tsv`, running the library's contamination check against both held-out
sets.

Then **state your hypothesis and a prediction**, before it trains. The hypothesis is the Final
Project's Path A in small: *"If I add back-translated data, then BLEU, chrF and COMET on my A8
test set will ..."*, finished with what you expect and why. The prediction says roughly how far
A13's BLEU will move from A9's, decoded greedily and with beam. The notebook stops until you
write both, and saves them to Drive for the write-up.

The cell after that is the long training, with A9's configuration and its validation and test
sets unchanged, so the added data is the only difference from A9. What it runs is A9's training,
so you can read what it prints: as many **epochs** (passes over the training set) as A9 ran; the
**training loss** falling as the model fits the pairs, and the **validation loss** on pairs it
never trains on, whose best value picks the checkpoint kept; a **learning rate** that rises over
the first steps (**warmup**) and then falls; **dropout**, which switches off a random tenth of
the network's units at each step so the model cannot lean on any one of them; and **gradient
clipping** at 1.0, which caps each update's size so one odd batch cannot throw the model off.

In [ ]:
from torchlingo.diagnostics import check_contamination


def combine(original, back, mono, held_out):
    """Return the A13 training set: the original pairs, then the back-translated ones.

    TODO: write it, as the markdown above describes. back[i] is the back-translator's English for
    mono[i]; held_out is the set of source sentences of your validation and test sets.
    """
    raise NotImplementedError("write combine, then run this cell again")


# A case whose answer you know: one good new pair, one empty, and one that is a test source.
toy = combine(pd.DataFrame({"src": ["hello"], "tgt": ["hola"]}),
              ["a dog", "", "a test sentence"], ["un perro", "vacío", "una frase"],
              held_out={"a test sentence"})
expected = [["hello", "hola"], ["a dog", "un perro"]]
assert list(toy.columns) == ["src", "tgt"], "keep the columns named src and tgt, in that order"
assert toy.values.tolist() == expected, (
    f"combine is not right yet: for this case it should return {expected}, and yours returned "
    f"{toy.values.tolist()}")

# The training set, checked against both held-out sets.
mono = (A8_DIR / "mono-used.txt").read_text(encoding="utf-8").splitlines()
back = (A8_DIR / "mono-used.bt-sample.en").read_text(encoding="utf-8").split("\n")[:len(mono)]
held_out = set(val_df.src) | set(test_df.src)
combined = combine(train_df[["src", "tgt"]], back, mono, held_out)
assert combined.iloc[:len(train_df)].values.tolist() == train_df[["src", "tgt"]].values.tolist(), (
    "the original pairs must come first, unchanged")
for name, frame in (("val", val_df), ("test", test_df)):
    result = check_contamination(frame.src, combined.src, name=name)
    print(result)
    assert result, "a held-out source sentence is in the new training set; see combine's TODO"
combined.to_csv(A8_DIR / "train-a13.tsv", sep="\t", index=False)
print(f"Written: train-a13.tsv, {len(combined):,} pairs: {len(train_df):,} original and "
      f"{len(combined) - len(train_df):,} back-translated.")

In [ ]:
HYPOTHESIS = ""   # <- "If I add back-translated data, then ... on my A8 test set will ..."
PREDICTION = ""   # <- how far A13's BLEU will move from A9's, greedy and beam, and why

assert HYPOTHESIS.strip(), "Write your hypothesis in HYPOTHESIS, then run this cell again."
assert HYPOTHESIS.strip().lower().startswith("if") and " then " in HYPOTHESIS.lower(), (
    'Write the hypothesis as "If ..., then ...": what you change, and what you expect it to do.')
assert PREDICTION.strip(), "Write your prediction in PREDICTION, then run this cell again."
save_note(hypothesis=HYPOTHESIS.strip(), prediction=PREDICTION.strip())
print("Hypothesis and prediction saved to a13-notes.json, for the write-up.")

In [ ]:
# The long training: English to your language, on the combined training set. If the session
# drops, run this again: an unfinished training resumes, a finished one is skipped.
train("train-a13.tsv", "val.tsv", "assignment-13", "best-a13")

## Step 5: score both systems, decoded two ways, and ask whether the difference is real

Two systems now translate the same A8/A9 test set: A9, and your A13 system. Each decodes it
**greedily and with beam search (k = 5)**, four outputs in all. Greedy takes the likeliest next
piece at each step; beam search keeps the five likeliest partial translations at each step and
returns the best finished one, which can find a better translation that starts with a less
likely word. That gives the comparison Lecture 10 asks for: whether A13 beats A9 when both
decode greedily, and when both decode with beam, separates what the added data bought from what
the decoder bought. Never compare one system's beam score with another's greedy one.

Each is scored with the BLEU A9 used and with chrF, which counts matching characters rather
than whole words, so it still moves when BLEU is near zero. **sacreBLEU's paired bootstrap** asks
whether the difference is real. It resamples your test set a thousand times, scores both systems
on each resample, and reports a 95% interval for each score and a *p* for A13 against A9:
roughly, how often a difference this large would appear by the luck of which sentences landed in
the test set. Below 0.05 is the usual line.

**Then say what the table says.** In the cell after it, set `REAL` to `True` or `False` for
greedy and for beam: is A13's BLEU difference from A9 real? The cell checks you against the *p*.

The beam half is slow, one sentence at a time. Everything is saved to Drive as it goes.

In [ ]:
from sacrebleu.metrics import BLEU, CHRF
from sacrebleu.significance import PairedTest

sources, references = list(test_df.src), list(test_df.tgt)
SYSTEMS = {"A9": "a9", "A13": "a13"}   # name: its best-*/ folder

# Both systems decode the test set both ways, each to its own file on Drive. A decode that is
# interrupted resumes when you run the cell again; beam goes one sentence at a time, so it is the
# slow half. If you retrain a model, delete its test-*.hyp files first, or they are reused.
hyps, hyp_files = {}, []
for decoder in ("greedy", "beam"):
    for system, name in SYSTEMS.items():
        path = A8_DIR / f"test-{name}-{decoder}.hyp"
        print(f"{system}, {decoder}: {path.name}")
        hyps[system, decoder] = translate_to_file(load_best(f"best-{name}"), sources, vocab, vocab,
                                                  path, decode_strategy=decoder, beam_size=5,
                                                  device=device, config=config, verbose=False)
        hyp_files.append(path.name)


def bootstrap(names, decoder):
    """sacreBLEU's paired bootstrap: each system's BLEU and chrF, a 95% interval, and p against
    the first system. Resampling the test set 1,000 times asks how often the difference would
    vanish on a different draw of sentences like these."""
    metrics = {"BLEU": BLEU(tokenize=TOKENIZE), "chrF": CHRF()}
    _, result = PairedTest([(n, hyps[n, decoder]) for n in names], metrics, [references],
                           test_type="bs", n_samples=1000)()
    bleu, chrf = (result[key] for key in result if key != "System")

    def number(x):
        return None if x is None else round(float(x), 4)   # None for the first system's p

    scored = {n: {"bleu": number(b.score), "bleu_ci": number(b.ci), "bleu_p": number(b.p_value),
                  "chrf": number(c.score), "chrf_ci": number(c.ci), "chrf_p": number(c.p_value)}
              for n, b, c in zip(names, bleu, chrf)}
    # sacreBLEU counts resampled differences strictly larger than the real one, so a system
    # identical to the first would come out at its smallest p, 0.001, "real". No difference is 1.
    for n in names[1:]:
        if hyps[n, decoder] == hyps[names[0], decoder]:
            scored[n].update(bleu_p=1.0, chrf_p=1.0)
    return scored


scores = {decoder: bootstrap(list(SYSTEMS), decoder) for decoder in ("greedy", "beam")}
save_note(scores=scores, hyp_files=hyp_files)

print(f"\n{'':8}{'system':<8}{'BLEU':>8}{'+/-':>7}{'chrF':>8}{'+/-':>7}   p against A9 (BLEU, chrF)")
for decoder in ("greedy", "beam"):
    for system, s in scores[decoder].items():
        against = "" if system == "A9" else f"   {s['bleu_p']:.3f}, {s['chrf_p']:.3f}"
        print(f"{decoder:<8}{system:<8}{s['bleu']:8.2f}{s['bleu_ci']:7.2f}{s['chrf']:8.2f}"
              f"{s['chrf_ci']:7.2f}{against}")

# The files COMET reads in Step 6, one sentence per line, the same line numbers as the .hyp files.
(A8_DIR / "test.src.txt").write_text("\n".join(sources) + "\n", encoding="utf-8")
(A8_DIR / "test.ref.txt").write_text("\n".join(references) + "\n", encoding="utf-8")

In [ ]:
REAL = {"greedy": None, "beam": None}   # <- for each: True if A13's BLEU difference from A9 is real, else False

assert None not in REAL.values(), (
    "Read the p column in the table above, then set True or False for greedy and for beam.")
p_values = {decoder: load_notes()["scores"][decoder]["A13"]["bleu_p"] for decoder in REAL}
for decoder, said in REAL.items():
    p = p_values[decoder]
    assert said == (p < 0.05), (
        f"For {decoder} you said {'real' if said else 'not real'}, but its p is {p:.3f}, which is "
        f"{'below' if p < 0.05 else 'not below'} 0.05. Look at that row again.")
save_note(real=REAL)
for decoder, said in REAL.items():
    print(f"{decoder}: {'real' if said else 'not real'} (p = {p_values[decoder]:.3f})")

## Step 6: COMET

The same reference-based COMET model you installed for A10, on all four translations of your test
set at once: A9 and A13, each greedy and beam. It needs no Hugging Face token. The first cell
installs it, the second scores. They are the last to run before the write-up, because installing
COMET can change packages the cells above rely on; if they give trouble, run them in a fresh
session on their own, after the setup cells. The second prints one system score per file, in
Step 5's order.

In [ ]:
# COMET asks for numpy older than 2, which would break packages Colab ships built for numpy 2;
# COMET itself runs on numpy 2. So its other requirements go in first, then COMET without them.
# pip may then print a red list of conflicts with other packages Colab ships; this uses none.
%pip install --quiet "entmax>=1.1,<2" "jsonargparse==3.13.1" "pytorch-lightning>=2,<3" "sacrebleu>=2,<3" "setuptools<81" "torchmetrics>=0.10.2,<0.11" "transformers>=4.17,<5"
%pip install --quiet --no-deps unbabel-comet
from importlib.metadata import version
if int(version("numpy").split(".")[0]) < 2:
    raise RuntimeError("An earlier COMET install replaced numpy in this session, which breaks COMET. "
                       "Runtime -> Disconnect and delete runtime, then run the setup cells and this one.")

In [ ]:
# All four translations of the test set, in the order Step 5 printed them.
HYP_FILES = " ".join(f'"{A8_DIR / name}"' for name in load_notes()["hyp_files"])
!comet-score -s "{A8_DIR}/test.src.txt" -r "{A8_DIR}/test.ref.txt" -t {HYP_FILES} --gpus 1 --quiet --only_system

## Step 7: the write-up

The cell below prints the numbers in the order the write-up asks for, the two test sentences
where back-translation helped most and the two where it hurt most, and saves `a13-rerun.json`.
It reads everything from Drive, so in a new session it needs only the setup cell. Answer these in
your write-up:

1. **Your hypothesis against the result.** Quote your "If ..., then ..." and your prediction,
   then give BLEU and chrF, with their intervals, for A9 and A13, and the COMET system scores
   from Step 6. Did the paired bootstrap find the difference real (p below 0.05)? Say in one
   sentence what the 95% interval means.
2. **Why sample the back-translations?** Find two sentences in your back-translated English
   (Step 3 printed some, sampled beside greedy) and say what in them would help or hurt a model
   learning from them. Edunov et al. (2018) found sampling helps in all but resource-poor
   settings: is your language one, and what does your result suggest?
3. **How much came from the data, and how much from decoding?** From the greedy and beam rows:
   how much of A13's change over A9 survives when both decode with beam, and how much does beam
   add to each system on its own? Lecture 10 said a comparison must pin decoding; say what
   pinning it showed here.
4. **Which side of each back-translated pair is real, and why is that the side that matters?**
   Use your back-translator's BLEU: say what its quality does to the pairs it made, and to what
   the new model learned from them.
5. **What makes your language low-resource, beyond its number of sentence pairs?** Say how many
   monolingual sentences you used and where they came from (the cell prints both), with a
   sentence on their domain against your A8 data's, and if fewer than your training pairs, why
   not more. Then name two other things, such as whether its spelling is standardized, or
   whether tools like Google Translate, NLLB or the tokenizer handle it well. Say which one
   limited A13 most.
6. **Where it helped and where it hurt.** For the four sentences below, add Google Translate's
   English for both outputs, so a reader without your language can judge them, and say what the
   back-translated data changed in each.
7. **How would someone rerun this?** Paste the contents of `a13-rerun.json`.

**Upload** to your folder in the Assignment shared folder: the training set `train-a13.tsv`,
`val.tsv` and `test.tsv` (unchanged from A9), every `test-*.hyp`, `a13-rerun.json`, the
write-up, and **this notebook**: File → Download → Download .ipynb, after it has run, so every
cell's output is in it.

In [ ]:
# Reads everything from Drive, so after a dropped session it needs only the setup cell above.
import importlib.metadata

notes = load_notes()
scores = notes["scores"]
combined = read_tsv("train-a13.tsv")
which = "spBLEU (flores200)" if NO_SPACES else "SacreBLEU, default tokenizer"

print("=== For the A13 write-up ===")
print(f"Hypothesis, written before training: {notes['hypothesis']}")
print(f"Prediction, written before training: {notes['prediction']}\n")
print("Monolingual sentences used, by source: "
      + ";  ".join(f"{source} {count:,}" for source, count in notes["mono_sources"].items()))
back = f"NLLB-200 ({notes['nllb_code']})" if notes["back_translator"] == "nllb" else "your reverse model"
print(f"Back-translator: {back};  its BLEU into English on your test set: {notes['bleu_reverse']:.2f}")
print(f"Training pairs: {len(train_df):,} original + {len(combined) - len(train_df):,} "
      f"back-translated by sampling = {len(combined):,}")
print(f"Configuration: A9's, unchanged, in every direction ({EPOCHS} epochs)\n")

print(f"BLEU is {which}. Each score is followed by its 95% interval (+/-) and, for A13, the "
      "paired bootstrap's p against A9 decoded the same way:")
for decoder in ("greedy", "beam"):
    for system, s in scores[decoder].items():
        against = "" if system == "A9" else f"   p = {s['bleu_p']:.3f} (BLEU), {s['chrf_p']:.3f} (chrF)"
        print(f"  {decoder:<7} {system:<5} BLEU {s['bleu']:6.2f} +/- {s['bleu_ci']:.2f}   "
              f"chrF {s['chrf']:6.2f} +/- {s['chrf_ci']:.2f}{against}")

# Two sentences where back-translation helped most and two where it hurt most: A13 against A9,
# both greedy, by each sentence's chrF.
sources, references = list(test_df.src), list(test_df.tgt)
hyp_a9 = (A8_DIR / "test-a9-greedy.hyp").read_text(encoding="utf-8").split("\n")[:len(sources)]
hyp_a13 = (A8_DIR / "test-a13-greedy.hyp").read_text(encoding="utf-8").split("\n")[:len(sources)]
change = sorted(
    (sacrebleu.sentence_chrf(h13, [r]).score - sacrebleu.sentence_chrf(h9, [r]).score, i)
    for i, (h9, h13, r) in enumerate(zip(hyp_a9, hyp_a13, references))
)
for title, picks in (("helped most", change[::-1][:2]), ("hurt most", change[:2])):
    print(f"\n=== Two sentences where back-translation {title}, by chrF ===")
    for delta, i in picks:
        print(f"\nchrF {delta:+.1f}\nSRC {sources[i]}\nREF {references[i]}\nA9  {hyp_a9[i]}\n"
              f"A13 {hyp_a13[i]}")

# Everything someone needs to rerun A13 and get the same numbers.
rerun = {
    "back_translator": back, "bt_method": notes["bt_method"], "top_k": notes.get("top_k"),
    "seed": SEED, "epochs": EPOCHS,
    "configuration": "A9's: d_model 512, 8 heads, 6+6 layers, d_ff 2048, batch 64, A9's SentencePiece",
    "scoring": f"greedy, and beam 5 with alpha 0.6; max length {MAX_DECODE}; {which} and chrF; "
               "paired bootstrap, 1,000 samples",
    "monolingual": notes["mono_sources"],
    "torchlingo": torchlingo.version_label(), "sacrebleu": importlib.metadata.version("sacrebleu"),
    "torch": torch.__version__,
}
(A8_DIR / "a13-rerun.json").write_text(json.dumps(rerun, indent=2, ensure_ascii=False) + "\n",
                                       encoding="utf-8")
print(f"\nSaved {A8_DIR / 'a13-rerun.json'}: paste it into the write-up (question 7).")

---
## When you finish: disconnect the GPU

**Runtime → Disconnect and delete runtime.** Colab charges compute units for every minute a GPU runtime stays connected, even when nothing is running, so a runtime left open after the work is done uses up your monthly units. Closing the browser tab does not end it right away. **Not while a training or back-translation cell is running**: ending the session stops it (it resumes in a new session, but the time in between is lost). Everything A13 makes is on Drive, so disconnecting at the end loses nothing.